In [2]:
import pandas as pd

In [3]:
df=pd.read_csv("../Dataset/IMDB Dataset.csv")

In [4]:
sentiment=df["sentiment"].map({"negative":0,"positive":1})

In [5]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
import gensim

gensim.utils.simple_preprocess(df["review"][0])

['one',
 'of',
 'the',
 'other',
 'reviewers',
 'has',
 'mentioned',
 'that',
 'after',
 'watching',
 'just',
 'oz',
 'episode',
 'you',
 'll',
 'be',
 'hooked',
 'they',
 'are',
 'right',
 'as',
 'this',
 'is',
 'exactly',
 'what',
 'happened',
 'with',
 'me',
 'br',
 'br',
 'the',
 'first',
 'thing',
 'that',
 'struck',
 'me',
 'about',
 'oz',
 'was',
 'its',
 'brutality',
 'and',
 'unflinching',
 'scenes',
 'of',
 'violence',
 'which',
 'set',
 'in',
 'right',
 'from',
 'the',
 'word',
 'go',
 'trust',
 'me',
 'this',
 'is',
 'not',
 'show',
 'for',
 'the',
 'faint',
 'hearted',
 'or',
 'timid',
 'this',
 'show',
 'pulls',
 'no',
 'punches',
 'with',
 'regards',
 'to',
 'drugs',
 'sex',
 'or',
 'violence',
 'its',
 'is',
 'hardcore',
 'in',
 'the',
 'classic',
 'use',
 'of',
 'the',
 'word',
 'br',
 'br',
 'it',
 'is',
 'called',
 'oz',
 'as',
 'that',
 'is',
 'the',
 'nickname',
 'given',
 'to',
 'the',
 'oswald',
 'maximum',
 'security',
 'state',
 'penitentary',
 'it',
 'focuses'

In [7]:
data=df["review"].apply(gensim.utils.simple_preprocess)

In [8]:
data

0        [one, of, the, other, reviewers, has, mentione...
1        [wonderful, little, production, br, br, the, f...
2        [thought, this, was, wonderful, way, to, spend...
3        [basically, there, family, where, little, boy,...
4        [petter, mattei, love, in, the, time, of, mone...
                               ...                        
49995    [thought, this, movie, did, down, right, good,...
49996    [bad, plot, bad, dialogue, bad, acting, idioti...
49997    [am, catholic, taught, in, parochial, elementa...
49998    [going, to, have, to, disagree, with, the, pre...
49999    [no, one, expects, the, star, trek, movies, to...
Name: review, Length: 50000, dtype: object

In [9]:
model=gensim.models.Word2Vec(
    window=10,
    workers=4,
    min_count=2
)

In [ ]:
model.build_vocab(data,progress_per=1000)

In [11]:
model.train(data,total_examples=model.corpus_count,epochs=10)

(84929767, 111764670)

In [12]:
model.wv.most_similar("movie")

[('film', 0.8764321208000183),
 ('flick', 0.7086038589477539),
 ('it', 0.6850559711456299),
 ('movies', 0.6030887961387634),
 ('show', 0.5446659326553345),
 ('thing', 0.5385378003120422),
 ('sequel', 0.523817241191864),
 ('series', 0.5010559558868408),
 ('storyline', 0.48710688948631287),
 ('really', 0.48426198959350586)]

In [13]:
model.wv.get_vector("good")

array([ 0.12874803, -0.6726723 ,  0.95396066,  2.668342  ,  0.7932939 ,
       -0.17118444, -3.1889741 , -1.9276845 , -3.219234  ,  2.3145497 ,
       -0.34365317,  3.7277958 , -3.2920122 , -0.65496796,  4.281823  ,
       -2.0426168 , -1.3710171 ,  0.58227414,  1.0030457 ,  2.4214268 ,
       -1.2792497 ,  0.99195117,  1.3210249 , -1.267073  ,  4.8594203 ,
       -2.7439353 ,  2.1082828 , -1.0384802 , -3.3928473 ,  2.1208532 ,
       -1.0132036 , -2.9704103 , -0.53960896, -0.6061354 ,  3.5145357 ,
        1.5614084 , -2.8325808 ,  0.07901794,  0.8186569 ,  0.36754242,
        1.9554281 , -1.5376767 , -2.1745114 , -1.7473791 , -2.8521867 ,
        0.49615714, -4.840749  ,  2.5798633 , -1.2373952 ,  1.4536867 ,
        0.92640316, -2.4484043 , -0.28179556,  1.0076345 ,  0.1428015 ,
        1.5477353 , -2.0149152 , -1.5586258 ,  0.4528371 , -2.367547  ,
        2.325366  , -2.6445837 ,  1.7434467 , -2.4802885 ,  1.6357039 ,
       -3.8850324 , -1.4628705 , -1.5138897 , -0.20511761, -4.32

In [14]:
import numpy as np
def makeFeatureVec(words, model, num_features):
    """
    Average all word vectors in a given list of words using a Word2Vec model.
    
    Args:
    - words: List of words (strings)
    - model: Trained Word2Vec model
    - num_features: Dimensionality of the Word2Vec vectors

    Returns:
    - featureVec: Averaged feature vector (1D NumPy array)
    """

    # Initialize a zero vector of the desired feature size
    featureVec = np.zeros((num_features,), dtype="float32")
    
    # Counter to track how many valid words we add
    nwords = 0

    # Set of all words in the model's vocabulary (faster lookup)
    index2word_set = set(model.wv.index_to_key)

    # Loop over each word
    for word in words:
        if word in index2word_set:
            nwords += 1
            featureVec = np.add(featureVec,model.wv[word])  # Add word vector to total. Same as result_operator = featureVec + wordVec
    if nwords > 0:
        featureVec = np.divide(featureVec, nwords) # same as result_operator = featureVec / nwords
    else:
        # Optionally, handle the case where no valid words are found
        # For example, you can return a zero vector or raise an exception
        pass  # Currently, it will return the zero vector as initialized
    # Final averaging
    
    return featureVec


In [15]:
featureVec = np.zeros((100,), dtype="float32")
featureVec

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
      dtype=float32)

In [16]:
index2word_set = set(model.wv.index_to_key)
index2word_set

{'attainment',
 'americana',
 'cubic',
 'avoid',
 'ashwar',
 'tolerating',
 'kemble',
 'medeiros',
 'nicks',
 'schwedt',
 'lassard',
 'buzzards',
 'greenquist',
 'tainting',
 'schwartzenegger',
 'paradoxes',
 'leyner',
 'verses',
 'giggity',
 'fires',
 'shaved',
 'spookhouse',
 'frail',
 'arabic',
 'motivate',
 'telemovie',
 'scathingly',
 'marengi',
 'trifled',
 'preffered',
 'wackos',
 'scab',
 'slops',
 'subtracts',
 'shootout',
 'ledgers',
 'shootouts',
 'abdullah',
 'ril',
 'smidgen',
 'blotch',
 'potty',
 'damage',
 'boils',
 'component',
 'propel',
 'divisions',
 'tggep',
 'hitlers',
 'volo',
 'nicolodi',
 'fax',
 'pods',
 'festive',
 'pilate',
 'sinise',
 'january',
 'sedate',
 'britches',
 'disobeyed',
 'indistinct',
 'whitney',
 'setting',
 'herge',
 'brontë',
 'sre',
 'streaming',
 'hayes',
 'depress',
 'elephantine',
 'occurring',
 'tricked',
 'holly',
 'beaded',
 'mojave',
 'lotto',
 'hateful',
 'fratboy',
 'retardedness',
 'afroreggae',
 'mammoth',
 'cash',
 'trusted',
 '

In [17]:
def getAvgFeatureVecs(reviews, model, num_features):
    """
    Apply makeFeatureVec() to a list of reviews.

    Args:
    - reviews: List of lists (each inner list is a list of words in one review)
    - model: Trained Word2Vec model
    - num_features: Dimensionality of word vectors

    Returns:
    - reviewFeatureVecs: 2D NumPy array, each row is an average vector
    """

    counter = 0  # For progress tracking
    # Preallocate 2D NumPy array for all review vectors
    reviewFeatureVecs = np.zeros((len(reviews), num_features), dtype="float32")

    for review in reviews:
        # Create average vector for this review
        reviewFeatureVecs[counter] = makeFeatureVec(review, model, num_features)
        counter += 1

    return reviewFeatureVecs


In [18]:
print(model.vector_size)


100


In [19]:
trainDataVecs = getAvgFeatureVecs(data, model, 100)

In [20]:
df["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [21]:
Y=df["sentiment"].map({"positive":1,"negative":0})

In [ ]:
trainDataVecs[1]    

array([-0.6055443 , -0.41298306, -0.00979633,  0.6605929 , -0.6248968 ,
        1.0136901 , -0.2703692 , -0.9445379 , -0.60904574,  0.38100606,
       -0.00368459,  0.6857794 ,  0.21793018, -0.10855814,  0.7002879 ,
        0.01513093,  0.23417045, -0.16778187, -0.81437844,  0.5162684 ,
        0.31943697,  0.37924764, -0.00209924, -0.05012336,  0.400956  ,
       -0.09646064,  0.23581974,  0.47143066, -0.23269865, -0.0150075 ,
        0.36559495, -0.539315  , -0.31538993, -0.35169148,  0.49814543,
        0.16629644, -0.97323817,  0.02700613,  0.19302697,  0.30367753,
        0.00618511,  0.05846292, -0.18763429,  0.19629133, -0.1901817 ,
        0.11683509, -0.27163333, -0.03857755, -0.34973046,  0.60163826,
       -0.13739741, -0.87605506, -0.49214652, -0.38077313,  0.50308204,
        0.35650587,  0.09069632, -0.4791291 , -0.44099364, -0.7280177 ,
       -0.42007565,  0.46810928, -0.46877232, -0.16442597,  1.0928953 ,
       -0.67604464,  0.73081386,  0.21988834,  0.13968034, -0.86

In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Assuming trainDataVecs is your feature matrix and labels is your target vector
X_train, X_test, y_train, y_test = train_test_split(trainDataVecs, Y, test_size=0.2, random_state=42)

# Initialize and train the Gaussian Naive Bayes classifier
gnb = RandomForestClassifier()
gnb.fit(X_train, y_train)

# Predict on the test set
y_pred = gnb.predict(X_test)

# Evaluate the classifier
accuracy = classification_report(y_test, y_pred)
print(accuracy)


              precision    recall  f1-score   support

           0       0.84      0.82      0.83      4961
           1       0.83      0.85      0.84      5039

    accuracy                           0.84     10000
   macro avg       0.84      0.84      0.84     10000
weighted avg       0.84      0.84      0.84     10000

